## Match to OpenAlex

### What we do

`huang_awards_pilot.csv` (925 rows, from notebook 03) is the single source of truth.
We left-join OpenAlex IDs onto it from two pre-existing matched files:
- **Huang conferences**: `huang_matched_openalex.csv` (previously matched via API)
- **ICWSM + JCDL**: `icwsm_jcdl_awards_raw.csv` (matched in notebook 01b)

No API calls needed. Output is always exactly the 925 pilot rows.

In [ ]:
import pandas as pd
import re

BASE = 'B:\\Semester 4 UU\\thesis-best-paper-trajectories\\data\\'

# ── Source of truth ───────────────────────────────────────────────────────
pilot = pd.read_csv(BASE + 'cleaned\\huang_awards_pilot.csv')
print(f'Pilot rows (source of truth): {len(pilot)}')

# ── Load pre-matched lookup tables ──────────────────────────────────────────
huang_matched = pd.read_csv(BASE + 'matched\\huang_matched_openalex.csv')
icwsm_jcdl   = pd.read_csv(BASE + 'raw\\icwsm_jcdl_awards_raw.csv')
icwsm_jcdl   = icwsm_jcdl.rename(columns={'openalex_title': 'oa_title'})
icwsm_jcdl['match_route'] = 'pre-matched'

# combine both lookup tables
MATCH_COLS = ['paper_title', 'year', 'openalex_id', 'doi', 'oa_title', 'authorships', 'match_route']
for col in MATCH_COLS:
    if col not in huang_matched.columns: huang_matched[col] = None
    if col not in icwsm_jcdl.columns:   icwsm_jcdl[col]   = None

lookup = pd.concat([huang_matched[MATCH_COLS], icwsm_jcdl[MATCH_COLS]], ignore_index=True)
# drop duplicates keeping first occurrence
lookup = lookup.drop_duplicates(subset=['paper_title', 'year'])
print(f'Lookup table size: {len(lookup)}')

# ── Left-join onto pilot (925 rows stays fixed) ─────────────────────────────
out = pilot.merge(
    lookup[MATCH_COLS],
    on=['paper_title', 'year'],
    how='left'
)

assert len(out) == len(pilot), f'Row count changed! {len(out)} != {len(pilot)}'

out.to_csv(BASE + 'matched\\huang_matched_openalex.csv', index=False)

matched = out['openalex_id'].notna().sum()
print(f"\n{'='*50}")
print(f"Total: {len(out)} | Matched: {matched} ({matched/len(out)*100:.1f}%)")
print(out.groupby('match_route')['openalex_id'].apply(lambda x: x.notna().sum()))


#### Check — flag low-confidence matches

In [ ]:
from rapidfuzz import fuzz
out['title_similarity'] = out.apply(
    lambda r: 100 if r['match_route'] in ('SS→DOI→OA', 'pre-matched')
    else fuzz.ratio(str(r['paper_title']).lower(), str(r['oa_title']).lower()),
    axis=1
)
out['low_confidence'] = out['title_similarity'] < 85
print(out['low_confidence'].sum(), 'flagged for manual review')


#### Manual Checkups

In [ ]:
unmatched = out[out['openalex_id'].isna()]
unmatched[['year','conference','paper_title']].to_csv(BASE + 'matched\\unmatched_manual.csv', index=False)
